
$\kappa=2,6$, con el mismo browniano y driving function $U_t=\sqrt\kappa B_t$. Código asistido con IA



In [ ]:
from pathlib import Path
import json
from IPython.display import display, Video, Markdown

CARPETA = Path.cwd()
KAPPAS = [2, 6, 20]
RECALIBRAR = False       
RENDERIZAR = False     
PRUEBA = True          
SOBRESCRIBIR = False
CODEC = "h264_nvenc"   


In [ ]:
for kappa in KAPPAS:
    nombre = f"SLE_kappa_{kappa}_4K.mp4"
    display(Markdown(f"**κ = {kappa}**"))
    if (CARPETA / nombre).exists():
        display(Video(filename=nombre, embed=False, width=960))


No encontrado: SLE_kappa_6_4K.mp4
No encontrado: SLE_kappa_20_4K.mp4


**κ = 2**

**κ = 6**

**κ = 20**

In [3]:
informe = json.loads((CARPETA / "calibracion_campo.json").read_text(encoding="utf-8"))
filas = ["| κ | N | P95 mapa | P95 dilatación log₂ |", "|---:|---:|---:|---:|"]
for r in informe["samples"]:
    filas.append(f"| {r['kappa']} | {r['N']:,} | {r['map_p95']:.3e} | {r['log2_derivative_p95']:.3e} |")
display(Markdown("\n".join(filas)))
for kappa in KAPPAS:
    archivo = CARPETA / f"SLE_kappa_{kappa}_4K.json"
    if archivo.exists():
        r = json.loads(archivo.read_text(encoding="utf-8"))
        print(f"κ={kappa}: {r['width']}×{r['height']}, {r['duration']} s, "
              f"{r['fps']} fps; cálculo y montaje: {r['total_seconds']/60:.2f} min")


κ=2: 3840×2160, 30.0 s, 24 fps; cálculo y montaje: 6.05 min


| κ | N | P95 mapa | P95 dilatación log₂ |
|---:|---:|---:|---:|
| 2 | 32,768 | 2.621e-05 | 5.523e-04 |
| 6 | 65,536 | 4.525e-05 | 1.214e-03 |
| 20 | 131,072 | 8.157e-05 | 2.903e-03 |

## Repetir las mediciones (opcional)
Requiere NVIDIA compatible y las dependencias de `requirements.txt`. Estas celdas no trabajan mientras `RECALIBRAR` y `RENDERIZAR` estén en `False`.

In [4]:
if RECALIBRAR:
    from calibrar import calibrar
    from verificar_campo import verificar
    rendimiento = calibrar()
    campo = verificar()


## Renderizar (opcional)
La malla temporal calibrada contiene 32.768, 65.536 y 131.072 pasos.

In [5]:
if RENDERIZAR:
    from renderizar import render
    for kappa in KAPPAS:
        sufijo = "_prueba" if PRUEBA else "_4K"
        destino = CARPETA / f"SLE_kappa_{kappa}{sufijo}.mp4"
        if destino.exists() and not SOBRESCRIBIR:
            print("Se conserva:", destino.name)
            continue
        render(kappa, width=960 if PRUEBA else 3840,
               height=540 if PRUEBA else 2160,
               fps=12 if PRUEBA else 24, duration=6 if PRUEBA else 30,
               preview=PRUEBA, codec=CODEC)


## Inspeccionar el conductor y la traza
La traza guardada son puntas de la aproximación por rendijas. El render omite conexiones de más de 8 píxeles 4K y deja puntos en esos saltos; no los sustituye por puentes rectos.

In [ ]:
import numpy as np

datos = {k: np.load(CARPETA / f"traza_k{k}.npz") for k in KAPPAS}
for k, d in datos.items():
    print(json.loads(str(d["stats"])))




{'kappa': 2, 'N': 32768, 'trace_samples': 3963, 'seconds': 2.414671099977568, 'max_tip_gap': 0.021705362195937324, 'gaps_over_8_pixels_at_4k': 2, 'max_y': 1.9320474790270565, 'min_x': -1.3789902346835017, 'max_x': 0.11993887129937454}
{'kappa': 6, 'N': 65536, 'trace_samples': 16630, 'seconds': 1.4095303999783937, 'max_tip_gap': 0.15047187691601635, 'gaps_over_8_pixels_at_4k': 315, 'max_y': 1.8195419619289863, 'min_x': -2.4713299145545764, 'max_x': 0.26724443761161043}
{'kappa': 20, 'N': 131072, 'trace_samples': 71454, 'seconds': 9.710666099999798, 'max_tip_gap': 0.15248036912639118, 'gaps_over_8_pixels_at_4k': 4629, 'max_y': 1.5464459593027664, 'min_x': -5.048330458471934, 'max_x': 0.6085417711306301}
